# 🧪 End-to-End LLM Stack Lab
## Observability, Evaluation, Safety & RAG with Production-Grade Tooling

**Lab Duration:** ~3 hours  
**Level:** Intermediate  
**Stack:** LangChain · LlamaIndex · Qdrant · NeMo Guardrails · Langfuse · DeepEval · Presidio · OpenRouter (Qwen 3 / any model)

---

### What You Will Build

```
┌─────────────────────────────────────────────────────────────────┐
│                        USER QUERY                               │
│                            │                                    │
│                    ┌───────▼────────┐                          │
│                    │  Presidio PII  │  ← Strip sensitive data  │
│                    └───────┬────────┘                          │
│                            │                                    │
│                    ┌───────▼────────┐                          │
│                    │ NeMo Guardrails│  ← Safety rails          │
│                    └───────┬────────┘                          │
│                            │                                    │
│              ┌─────────────▼──────────────┐                    │
│              │  LlamaIndex RAG Pipeline   │  ← Qdrant vector DB│
│              └─────────────┬──────────────┘                    │
│                            │                                    │
│              ┌─────────────▼──────────────┐                    │
│              │   LangChain Agent (Qwen)   │  ← OpenRouter      │
│              └─────────────┬──────────────┘                    │
│                            │                                    │
│              ┌─────────────▼──────────────┐                    │
│              │      Langfuse Tracing      │  ← Observability   │
│              └─────────────┬──────────────┘                    │
│                            │                                    │
│              ┌─────────────▼──────────────┐                    │
│              │    DeepEval Evaluation     │  ← Metrics & CI    │
│              └────────────────────────────┘                    │
└─────────────────────────────────────────────────────────────────┘
```

### Learning Objectives
1. Set up a full LLM stack from scratch with open-source models
2. Implement PII detection and removal before queries reach the LLM
3. Apply NeMo Guardrails for input/output safety policies
4. Build a RAG pipeline over custom documents using LlamaIndex + Qdrant
5. Trace every LLM call end-to-end with Langfuse
6. Evaluate RAG quality with DeepEval (faithfulness, relevancy, etc.)
7. Run automated evaluation test suites

### Model Options (choose one)
| Option | Provider | Model | Notes |
|--------|----------|-------|-------|
| A | OpenRouter | `qwen/qwen3-8b` | Free tier available |
| B | Alibaba DashScope | `qwen-turbo` | Fast & cheap |
| C | Local Ollama | `qwen2.5:7b` | Fully offline |


---
## 📦 Section 0: Installation

Run once. This installs everything needed for the lab.

> **Note:** If you're on Google Colab, restart the runtime after this cell completes.

In [16]:
# Install all dependencies
# This may take 2-3 minutes

%pip install -q \
    langchain==0.3.7 \
    langchain-openai==0.2.8 \
    langchain-community==0.3.7 \
    llama-index==0.11.20 \
    llama-index-vector-stores-qdrant==0.3.3 \
    llama-index-embeddings-huggingface==0.3.1 \
    qdrant-client==1.12.1 \
    langfuse==2.53.5 \
    deepeval==1.4.8 \
    presidio-analyzer==2.2.354 \
    presidio-anonymizer==2.2.354 \
    spacy==3.8.2 \
    nemoguardrails==0.10.1 \
    openai==1.55.3 \
    httpx==0.27.2 \
    python-dotenv==1.0.1 \
    datasets==3.1.0 \
    sentence-transformers==3.3.1

import sys
import subprocess
import spacy

model_name = "en_core_web_lg"

try:
    spacy.load(model_name)
    print(f"✅ spaCy model '{model_name}' is already installed.")
except Exception:
    print(f"Downloading spaCy model '{model_name}'...")
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", model_name],
        check=True
    )
    print(f"✅ spaCy model '{model_name}' installed successfully.")

print("✅ All dependencies installed successfully!")


ERROR: Ignored the following versions that require a different python version: 0.0.1 Requires-Python >=3.8.1,<3.12; 0.1.0 Requires-Python >=3.8.1,<3.12; 0.1.1 Requires-Python >=3.8.1,<3.12; 0.1.2 Requires-Python >=3.8.1,<3.12; 0.1.5 Requires-Python >=3.9,<3.13; 0.1.6 Requires-Python >=3.9,<3.13; 0.10.0 Requires-Python >=3.8.1,<3.12; 0.10.1 Requires-Python >=3.8.1,<3.12; 0.10.11 Requires-Python >=3.8.1,<3.12; 0.10.12 Requires-Python >=3.8.1,<3.12; 0.10.3 Requires-Python >=3.8.1,<3.12; 0.10.4 Requires-Python >=3.8.1,<3.12; 0.2.0 Requires-Python >=3.9,<3.13; 0.2.1 Requires-Python >=3.9,<3.13; 0.2.10 Requires-Python >=3.9,<3.13; 0.2.11 Requires-Python >=3.9,<3.13; 0.2.12 Requires-Python >=3.9,<3.13; 0.2.13 Requires-Python >=3.9,<3.13; 0.2.14 Requires-Python >=3.9,<3.13; 0.2.15 Requires-Python >=3.9,<3.13; 0.2.16 Requires-Python >=3.9,<3.13; 0.2.17 Requires-Python >=3.9,<3.13; 0.2.2 Requires-Python >=3.9,<3.13; 0.2.3 Requires-Python >=3.9,<3.13; 0.2.4 Requires-Python >=3.9,<3.13; 0.2.5 Requ

Note: you may need to restart the kernel to use updated packages.
✅ spaCy model 'en_core_web_lg' is already installed.
✅ All dependencies installed successfully!


---
## ⚙️ Section 1: Configuration

Choose your model backend and set your API keys below.

In [17]:
import os

# ─────────────────────────────────────────────
# CHOOSE YOUR BACKEND  (set exactly one to True)
# ─────────────────────────────────────────────
USE_OPENROUTER = True    # OpenRouter with Qwen
USE_DASHSCOPE  = False   # Alibaba DashScope
USE_OLLAMA     = False   # Local Ollama

# ─────────────────────────────────────────────
# API KEYS  (replace with your actual keys)
# ─────────────────────────────────────────────

# Option A: OpenRouter  →  https://openrouter.ai/keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Option B: Alibaba DashScope  →  https://dashscope.aliyuncs.com
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY");

# Langfuse  →  https://cloud.langfuse.com  (free)
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_BASE_URL")  # or your self-hosted URL

# DeepEval  →  https://app.confident-ai.com  (free)
# (DeepEval can also run fully local — we'll configure that below)
DEEPEVAL_API_KEY = "YOUR_DEEPEVAL_KEY_OR_LEAVE_BLANK"

# ─────────────────────────────────────────────
# RESOLVE ACTIVE BACKEND
# ─────────────────────────────────────────────
if USE_OPENROUTER:
    LLM_BASE_URL    = "https://openrouter.ai/api/v1"
    LLM_API_KEY     = OPENROUTER_API_KEY
    LLM_MODEL_NAME  = "qwen/qwen3-8b"          # free on OpenRouter
    EMBED_MODEL     = "BAAI/bge-small-en-v1.5" # local embedding
    print("🌐 Backend: OpenRouter → qwen/qwen3-8b")

elif USE_DASHSCOPE:
    LLM_BASE_URL    = "https://dashscope.aliyuncs.com/compatible-mode/v1"
    LLM_API_KEY     = DASHSCOPE_API_KEY
    LLM_MODEL_NAME  = "qwen-turbo"
    EMBED_MODEL     = "BAAI/bge-small-en-v1.5"
    print("☁️  Backend: Alibaba DashScope → qwen-turbo")

elif USE_OLLAMA:
    LLM_BASE_URL    = "http://localhost:11434/v1"
    LLM_API_KEY     = "ollama"                 # placeholder
    LLM_MODEL_NAME  = "qwen2.5:7b"
    EMBED_MODEL     = "BAAI/bge-small-en-v1.5"
    print("🖥️  Backend: Local Ollama → qwen2.5:7b")
    print("   Make sure ollama is running: ollama serve")
    print("   Pull model first:           ollama pull qwen2.5:7b")

# Set env vars used by libraries
# os.environ["OPENAI_API_KEY"]       = LLM_API_KEY
# os.environ["OPENAI_API_BASE"]      = LLM_BASE_URL
# os.environ["LANGFUSE_PUBLIC_KEY"]  = LANGFUSE_PUBLIC_KEY
# os.environ["LANGFUSE_SECRET_KEY"]  = LANGFUSE_SECRET_KEY
# os.environ["LANGFUSE_HOST"]        = LANGFUSE_HOST
if DEEPEVAL_API_KEY:
    os.environ["CONFIDENT_API_KEY"] = DEEPEVAL_API_KEY

print("✅ Configuration complete.")

🌐 Backend: OpenRouter → qwen/qwen3-8b
✅ Configuration complete.


---
## 🔍 Section 2: PII Detection & Anonymization with Presidio

Before any user query reaches the LLM, we scan for Personally Identifiable Information (PII) and replace it with safe placeholders. This is a critical compliance layer.

**Entities detected:** Names, email addresses, phone numbers, credit card numbers, SSNs, IP addresses, locations, and more.

In [18]:
!pip install presidio-analyzer presidio-anonymizer
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Initialize Presidio engines
analyzer  = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def detect_pii(text: str, language: str = "en") -> list:
    """
    Detect PII entities in text.
    Returns a list of detected entities with their positions and types.
    """
    results = analyzer.analyze(text=text, language=language)
    return results


def anonymize_text(text: str, language: str = "en") -> dict:
    """
    Detect and replace PII with type-labelled placeholders.
    Returns dict with 'anonymized_text' and 'entities_found'.
    """
    analysis_results = analyzer.analyze(text=text, language=language)

    if not analysis_results:
        return {"anonymized_text": text, "entities_found": []}

    # Replace each entity type with a descriptive placeholder
    operators = {
        "DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"}),
        "PERSON":       OperatorConfig("replace", {"new_value": "<PERSON>"}),
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
        "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
        "CREDIT_CARD":  OperatorConfig("replace", {"new_value": "<CREDIT_CARD>"}),
        "US_SSN":       OperatorConfig("replace", {"new_value": "<SSN>"}),
        "IP_ADDRESS":   OperatorConfig("replace", {"new_value": "<IP_ADDRESS>"}),
        "LOCATION":     OperatorConfig("replace", {"new_value": "<LOCATION>"}),
    }

    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=analysis_results,
        operators=operators
    )

    entities_found = [
        {"type": r.entity_type, "score": round(r.score, 2), "text": text[r.start:r.end]}
        for r in analysis_results
    ]

    return {
        "anonymized_text": anonymized.text,
        "entities_found": entities_found
    }


print("✅ Presidio engines loaded.")
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Initialize Presidio engines
analyzer  = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def detect_pii(text: str, language: str = "en") -> list:
    """
    Detect PII entities in text.
    Returns a list of detected entities with their positions and types.
    """
    results = analyzer.analyze(text=text, language=language)
    return results


def anonymize_text(text: str, language: str = "en") -> dict:
    """
    Detect and replace PII with type-labelled placeholders.
    Returns dict with 'anonymized_text' and 'entities_found'.
    """
    analysis_results = analyzer.analyze(text=text, language=language)

    if not analysis_results:
        return {"anonymized_text": text, "entities_found": []}

    # Replace each entity type with a descriptive placeholder
    operators = {
        "DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"}),
        "PERSON":       OperatorConfig("replace", {"new_value": "<PERSON>"}),
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
        "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
        "CREDIT_CARD":  OperatorConfig("replace", {"new_value": "<CREDIT_CARD>"}),
        "US_SSN":       OperatorConfig("replace", {"new_value": "<SSN>"}),
        "IP_ADDRESS":   OperatorConfig("replace", {"new_value": "<IP_ADDRESS>"}),
        "LOCATION":     OperatorConfig("replace", {"new_value": "<LOCATION>"}),
    }

    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=analysis_results,
        operators=operators
    )

    entities_found = [
        {"type": r.entity_type, "score": round(r.score, 2), "text": text[r.start:r.end]}
        for r in analysis_results
    ]

    return {
        "anonymized_text": anonymized.text,
        "entities_found": entities_found
    }


print("✅ Presidio engines loaded.")

✅ Presidio engines loaded.
✅ Presidio engines loaded.


In [19]:
# ── Define PII Analyzer + Anonymizer ──────────────────────────────────────────

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Initialize Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()


def anonymize_text(text: str):
    """
    Detects and anonymizes PII from input text using Microsoft Presidio.

    Returns:
        {
            "original_text": str,
            "anonymized_text": str,
            "entities_found": list
        }
    """

    # Analyze text for PII
    analyzer_results = analyzer.analyze(
        text=text,
        language="en"
    )

    # Configure anonymization behavior
    operators = {
        "DEFAULT": OperatorConfig(
            "replace",
            {
                "new_value": "<PII>"
            }
        ),
        "PERSON": OperatorConfig(
            "replace",
            {
                "new_value": "<PERSON>"
            }
        ),
        "EMAIL_ADDRESS": OperatorConfig(
            "replace",
            {
                "new_value": "<EMAIL>"
            }
        ),
        "PHONE_NUMBER": OperatorConfig(
            "replace",
            {
                "new_value": "<PHONE_NUMBER>"
            }
        ),
        "IP_ADDRESS": OperatorConfig(
            "replace",
            {
                "new_value": "<IP_ADDRESS>"
            }
        ),
        "US_SSN": OperatorConfig(
            "replace",
            {
                "new_value": "<US_SSN>"
            }
        ),
        "CREDIT_CARD": OperatorConfig(
            "replace",
            {
                "new_value": "<CREDIT_CARD>"
            }
        )
    }

    # Anonymize detected entities
    anonymized_result = anonymizer.anonymize(
        text=text,
        analyzer_results=analyzer_results,
        operators=operators
    )

    # Format detected entities for easier inspection
    entities_found = []

    for result in analyzer_results:
        entities_found.append({
            "entity_type": result.entity_type,
            "start": result.start,
            "end": result.end,
            "score": round(result.score, 4),
            "text": text[result.start:result.end]
        })

    return {
        "original_text": text,
        "anonymized_text": anonymized_result.text,
        "entities_found": entities_found
    }

In [20]:
# ── Test PII Detection ───────────────────────────────────────────────────────

test_queries = [
    "My name is John Smith and my email is john.smith@company.com. Can you help?",
    "Call me at +971-553661755 or reach me at 192.168.1.100",
    "My SSN is 123-45-6789 and my card ends in 4242 4242 4242 4242",
    "What is the capital of France?"  # No PII — should pass through unchanged
]

print("=" * 65)
print("PII DETECTION & ANONYMIZATION TEST")
print("=" * 65)

for query in test_queries:
    result = anonymize_text(query)

    print(f"\n📥 Input:     {query}")
    print(f"📤 Sanitized: {result['anonymized_text']}")

    if result["entities_found"]:
        print(f"🔎 Entities:  {result['entities_found']}")
    else:
        print("✅ No PII detected.")

    print("-" * 65)

PII DETECTION & ANONYMIZATION TEST

📥 Input:     My name is John Smith and my email is john.smith@company.com. Can you help?
📤 Sanitized: My name is <PERSON> and my email is <EMAIL>. Can you help?
🔎 Entities:  [{'entity_type': 'EMAIL_ADDRESS', 'start': 38, 'end': 60, 'score': 1.0, 'text': 'john.smith@company.com'}, {'entity_type': 'PERSON', 'start': 11, 'end': 21, 'score': 0.85, 'text': 'John Smith'}, {'entity_type': 'URL', 'start': 38, 'end': 45, 'score': 0.5, 'text': 'john.sm'}, {'entity_type': 'URL', 'start': 49, 'end': 60, 'score': 0.5, 'text': 'company.com'}]
-----------------------------------------------------------------

📥 Input:     Call me at +971-553661755 or reach me at 192.168.1.100
📤 Sanitized: Call me at <PHONE_NUMBER> or reach me at <IP_ADDRESS>
🔎 Entities:  [{'entity_type': 'IP_ADDRESS', 'start': 41, 'end': 54, 'score': 0.6, 'text': '192.168.1.100'}, {'entity_type': 'PHONE_NUMBER', 'start': 11, 'end': 25, 'score': 0.4, 'text': '+971-553661755'}, {'entity_type': 'PHONE

In [21]:
# ── UAE Phone Number PII Detection + Anonymization in Single Block ───────────

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


# Initialize Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()


# UAE phone number patterns
uae_phone_patterns = [
    Pattern(
        name="uae_mobile_international",
        regex=r"(?:\+971|00971)[\s\-]?(?:5[024568])[\s\-]?\d{3}[\s\-]?\d{4}",
        score=0.90,
    ),
    Pattern(
        name="uae_mobile_local",
        regex=r"\b05[024568][\s\-]?\d{3}[\s\-]?\d{4}\b",
        score=0.90,
    ),
    Pattern(
        name="uae_landline_international",
        regex=r"(?:\+971|00971)[\s\-]?[234679][\s\-]?\d{3}[\s\-]?\d{4}",
        score=0.80,
    ),
    Pattern(
        name="uae_landline_local",
        regex=r"\b0[234679][\s\-]?\d{3}[\s\-]?\d{4}\b",
        score=0.80,
    ),
]


# Create UAE phone recognizer
uae_phone_recognizer = PatternRecognizer(
    supported_entity="UAE_PHONE_NUMBER",
    patterns=uae_phone_patterns,
    context=[
        "phone",
        "mobile",
        "call",
        "contact",
        "number",
        "uae",
        "dubai",
        "abu dhabi",
        "sharjah",
        "emirates",
        "whatsapp",
    ],
)


# Register custom recognizer
analyzer.registry.add_recognizer(uae_phone_recognizer)


def anonymize_text(text: str):
    """
    Detects and anonymizes PII from input text using Microsoft Presidio.
    Includes custom UAE phone number detection.
    """

    analyzer_results = analyzer.analyze(
        text=text,
        language="en",
        entities=[
            "PERSON",
            "EMAIL_ADDRESS",
            "PHONE_NUMBER",
            "UAE_PHONE_NUMBER",
            "IP_ADDRESS",
            "US_SSN",
            "CREDIT_CARD",
        ],
    )

    operators = {
        "DEFAULT": OperatorConfig(
            "replace",
            {"new_value": "<PII>"}
        ),
        "PERSON": OperatorConfig(
            "replace",
            {"new_value": "<PERSON>"}
        ),
        "EMAIL_ADDRESS": OperatorConfig(
            "replace",
            {"new_value": "<EMAIL>"}
        ),
        "PHONE_NUMBER": OperatorConfig(
            "replace",
            {"new_value": "<PHONE_NUMBER>"}
        ),
        "UAE_PHONE_NUMBER": OperatorConfig(
            "replace",
            {"new_value": "<UAE_PHONE_NUMBER>"}
        ),
        "IP_ADDRESS": OperatorConfig(
            "replace",
            {"new_value": "<IP_ADDRESS>"}
        ),
        "US_SSN": OperatorConfig(
            "replace",
            {"new_value": "<US_SSN>"}
        ),
        "CREDIT_CARD": OperatorConfig(
            "replace",
            {"new_value": "<CREDIT_CARD>"}
        ),
    }

    anonymized_result = anonymizer.anonymize(
        text=text,
        analyzer_results=analyzer_results,
        operators=operators,
    )

    entities_found = []

    for result in analyzer_results:
        entities_found.append({
            "entity_type": result.entity_type,
            "start": result.start,
            "end": result.end,
            "score": round(result.score, 4),
            "text": text[result.start:result.end],
        })

    return {
        "original_text": text,
        "anonymized_text": anonymized_result.text,
        "entities_found": entities_found,
    }


# ── Test PII Detection ───────────────────────────────────────────────────────

test_queries = [
    "My name is John Smith and my email is john.smith@company.com. Can you help?",
    "Call me at +971-553661755 or reach me at 192.168.1.100",
    "My UAE mobile is +971 55 366 1755.",
    "You can WhatsApp me on 0553661755.",
    "Office number is 02 123 4567.",
    "Dubai landline is +971 4 123 4567.",
    "My SSN is 123-45-6789 and my card ends in 4242 4242 4242 4242",
    "What is the capital of France?"
]

print("=" * 65)
print("PII DETECTION & ANONYMIZATION TEST")
print("=" * 65)

for query in test_queries:
    result = anonymize_text(query)

    print(f"\n📥 Input:     {query}")
    print(f"📤 Sanitized: {result['anonymized_text']}")

    if result["entities_found"]:
        print(f"🔎 Entities:  {result['entities_found']}")
    else:
        print("✅ No PII detected.")

    print("-" * 65)

PII DETECTION & ANONYMIZATION TEST

📥 Input:     My name is John Smith and my email is john.smith@company.com. Can you help?
📤 Sanitized: My name is <PERSON> and my email is <EMAIL>. Can you help?
🔎 Entities:  [{'entity_type': 'EMAIL_ADDRESS', 'start': 38, 'end': 60, 'score': 1.0, 'text': 'john.smith@company.com'}, {'entity_type': 'PERSON', 'start': 11, 'end': 21, 'score': 0.85, 'text': 'John Smith'}]
-----------------------------------------------------------------

📥 Input:     Call me at +971-553661755 or reach me at 192.168.1.100
📤 Sanitized: Call me at <UAE_PHONE_NUMBER> or reach me at <IP_ADDRESS>
🔎 Entities:  [{'entity_type': 'UAE_PHONE_NUMBER', 'start': 11, 'end': 25, 'score': 0.9, 'text': '+971-553661755'}, {'entity_type': 'IP_ADDRESS', 'start': 41, 'end': 54, 'score': 0.6, 'text': '192.168.1.100'}, {'entity_type': 'PHONE_NUMBER', 'start': 11, 'end': 25, 'score': 0.4, 'text': '+971-553661755'}, {'entity_type': 'PHONE_NUMBER', 'start': 41, 'end': 54, 'score': 0.4, 'text': '192.

---
## 🛡️ Section 3: Safety Rails with NeMo Guardrails

NeMo Guardrails lets us define **colang** policies that govern:
- What topics the bot should refuse
- What tone/format responses must follow
- How to handle off-topic or harmful queries

We'll define policies for a **corporate knowledge assistant** use case.

In [22]:
import os

# Create config directory for NeMo Guardrails
os.makedirs("guardrails_config", exist_ok=True)

# ── colang policy file ────────────────────────────────────────────────────────
COLANG_CONTENT = """
# Define what topics are off-limits
define user ask harmful question
    "How do I hack into a system?"
    "Tell me how to make malware"
    "How do I bypass security?"
    "Give me someone's personal data"

define bot refuse harmful question
    "I'm sorry, but I can't assist with that request. I'm here to help with
    legitimate knowledge base questions only."

# Define off-topic handling
define user ask off topic
    "What is the weather today?"
    "Tell me a joke"
    "Who won the football game?"
    "Write me a poem"

define bot handle off topic
    "I'm a knowledge assistant focused on company documentation. I can't help
    with that, but I'm happy to answer questions about our products or policies."

# Core flow
define flow
    user ask harmful question
    bot refuse harmful question

define flow
    user ask off topic
    bot handle off topic
"""

# ── config.yml ────────────────────────────────────────────────────────────────
CONFIG_CONTENT = f"""
models:
  - type: main
    engine: openai
    model: {LLM_MODEL_NAME}
    parameters:
      base_url: {LLM_BASE_URL}
      api_key: {LLM_API_KEY}

instructions:
  - type: general
    content: |
      You are a helpful corporate knowledge assistant. You answer questions
      about company policies, products, and documentation. You are professional,
      concise, and accurate. You do not share personal data, engage in harmful
      activities, or go off-topic.

rails:
  input:
    flows:
      - check jailbreak
  output:
    flows:
      - check output for sensitive data
"""

with open("guardrails_config/main.co", "w") as f:
    f.write(COLANG_CONTENT)

with open("guardrails_config/config.yml", "w") as f:
    f.write(CONFIG_CONTENT)

print("✅ NeMo Guardrails config files written.")

✅ NeMo Guardrails config files written.


In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install nemoguardrails==0.10.1

In [27]:
%pip show nemoguardrails
%pip install --upgrade pip setuptools wheel

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install numpy==1.26.4 --only-binary=:all:

Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Could not find a version that satisfies the requirement numpy==1.26.4 (from versions: 2.1.0, 2.1.1, 2.1.2, 2.1.3, 2.2.0, 2.2.1, 2.2.2, 2.2.3, 2.2.4, 2.2.5, 2.2.6, 2.3.0, 2.3.1, 2.3.2, 2.3.3, 2.3.4, 2.3.5, 2.4.0rc1, 2.4.1, 2.4.2, 2.4.3, 2.4.4, 2.4.5, 2.4.6, 2.5.0rc1)
ERROR: No matching distribution found for numpy==1.26.4


In [ ]:
%pip install nemoguardrails==0.10.1 --no-cache-dir

---
## 🧪 Section 6: Evaluation with DeepEval

DeepEval provides automated metrics to evaluate RAG quality. We'll measure:

| Metric | What it measures |
|--------|------------------|
| **Faithfulness** | Does the answer only use facts from the retrieved context? |
| **Answer Relevancy** | Does the answer actually address the question asked? |
| **Contextual Precision** | Are all retrieved chunks relevant to the question? |
| **Contextual Recall** | Does the context contain enough info to answer the question? |
| **Hallucination** | Does the answer contain facts not in the context? |

In [ ]:
%pip install deepeval

In [40]:
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    HallucinationMetric,
)
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI


# ── Custom judge model pointing to our backend ────────────────────────────────
# DeepEval needs an LLM to judge answers. We point it to the same backend.

class CustomJudgeLLM(DeepEvalBaseLLM):
    """
    Custom DeepEval judge that uses our OpenRouter / DashScope / Ollama backend
    instead of requiring an OpenAI key.
    """

    def __init__(self):
        self.client = OpenAI(
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
            max_retries=1000
        )
        self.model_name = LLM_MODEL_NAME

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return self.model_name


judge_llm = CustomJudgeLLM()
print(f"✅ DeepEval judge model configured: {LLM_MODEL_NAME}")

✅ DeepEval judge model configured: qwen/qwen3-8b


In [34]:
# ── Build evaluation dataset ──────────────────────────────────────────────────
# Format: (question, expected_answer, retrieved_context)

EVAL_DATASET = [
    {
        "input": "What is the refund timeframe for standard returns?",
        "expected_output": "Items must be returned within 30 days, and the refund is processed within 5-10 business days of receiving the returned item.",
        "context": [
            "Our company offers a 30-day return policy for all products. "
            "Refund is processed within 5-10 business days of receiving the return."
        ]
    },
    {
        "input": "What encryption standard does the platform use for data at rest?",
        "expected_output": "The platform uses AES-256 encryption for data at rest.",
        "context": [
            "Customer data is encrypted in transit (TLS 1.3) and at rest (AES-256). "
            "Production data is never used for model training."
        ]
    },
    {
        "input": "How much does ConnectFlow cost?",
        "expected_output": "ConnectFlow costs $1,200 per month as a base price, plus $50 per additional connector.",
        "context": [
            "ConnectFlow: Enterprise integration platform supporting 200+ connectors. "
            "Pricing: $1,200/month base + $50 per additional connector."
        ]
    },
    {
        "input": "How long is the hypercare support period after go-live?",
        "expected_output": "Hypercare support is provided for the first 30 days after the production launch.",
        "context": [
            "Week 4: Go-Live — Production environment handover. "
            "Hypercare support for first 30 days post-launch."
        ]
    },
    {
        "input": "Does the company offer a HIPAA Business Associate Agreement?",
        "expected_output": "Yes, a HIPAA BAA is available upon request.",
        "context": [
            "HIPAA BAA available on request. "
            "SOC 2 Type II, ISO 27001, GDPR compliant."
        ]
    },
]

print(f"✅ Evaluation dataset ready: {len(EVAL_DATASET)} test cases")

✅ Evaluation dataset ready: 5 test cases


In [36]:
EVAL_DATASET = [
    {
        "input": "What is LangGraph?",
        "actual_output": "LangGraph is a framework for building AI workflows.",
        "expected_output": "LangGraph is a framework for building stateful AI agents and workflows."
    },
    {
        "input": "What is ChromaDB?",
        "actual_output": "ChromaDB is a vector database.",
        "expected_output": "ChromaDB is an open-source vector database used in RAG applications."
    }
]

In [37]:
from deepeval.test_case import LLMTestCase

test_cases = []

for item in EVAL_DATASET:

    # Simulated AI output
    actual_output = item["actual_output"]

    tc = LLMTestCase(
        input=item["input"],
        actual_output=actual_output,
        expected_output=item["expected_output"]
    )

    test_cases.append(tc)

print(f"✅ {len(test_cases)} test cases ready.")

✅ 2 test cases ready.


In [41]:
# ============================================================
# 1. IMPORTS
# ============================================================

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    HallucinationMetric,
)

# ============================================================
# 2. JUDGE MODEL
# ============================================================
# Replace with your judge model if configured.
# Example:
#
# from deepeval.models import GPTModel
# judge_llm = GPTModel(model="gpt-4o")
#
# For now assume judge_llm already exists.

# ============================================================
# 3. SAMPLE DATASET
# ============================================================

EVAL_DATASET = [
    {
        "input": "What is Python?",
        "actual_output": (
            "Python is a high-level programming language "
            "used for web development, AI, automation, and data science."
        ),
        "expected_output": (
            "Python is a high-level programming language."
        ),
    },
    {
        "input": "What is LangGraph?",
        "actual_output": (
            "LangGraph is a framework for building stateful "
            "AI agents and workflows."
        ),
        "expected_output": (
            "LangGraph is a framework used to build "
            "stateful AI workflows and agents."
        ),
    },
    {
        "input": "What is ChromaDB?",
        "actual_output": (
            "ChromaDB is an open-source vector database "
            "commonly used in RAG applications."
        ),
        "expected_output": (
            "ChromaDB is a vector database for storing "
            "and searching embeddings."
        ),
    },
]

# ============================================================
# 4. BUILD TEST CASES
# ============================================================

test_cases = []

for item in EVAL_DATASET:

    tc = LLMTestCase(
        input=item["input"],
        actual_output=item["actual_output"],
        expected_output=item["expected_output"],
    )

    test_cases.append(tc)

print(f"✅ Created {len(test_cases)} test cases")

# ============================================================
# 5. DEFINE METRICS
# ============================================================

THRESHOLD = 0.7

metrics = [

    AnswerRelevancyMetric(
        threshold=THRESHOLD,
        model=judge_llm,
        include_reason=True,
    ),

    HallucinationMetric(
        threshold=0.3,
        model=judge_llm,
        include_reason=True,
    ),
]

# ============================================================
# 6. RUN EVALUATION
# ============================================================

print("=" * 60)
print("RUNNING DEEPEVAL")
print("=" * 60)

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
)

print("\n✅ Evaluation Complete")

# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

for result in results.test_results:

    print("\n" + "=" * 60)

    print("Input:")
    print(result.input)

    print("\nActual Output:")
    print(result.actual_output)

    print("\nExpected Output:")
    print(result.expected_output)

    print("\nMetric Results:")

    for metric in result.metrics_data:

        print(f"\nMetric: {metric.name}")
        print(f"Score: {metric.score}")
        print(f"Passed: {metric.success}")

        if metric.reason:
            print(f"Reason: {metric.reason}")

✅ Created 3 test cases
RUNNING DEEPEVAL


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen/qwen3-8b, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Hallucination Metric! (using qwen/qwen3-8b, strict=False, async_mode=True)...

d:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 8192 tokens, but can only afford 434. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 8192 tokens, but can only afford 382. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3C7fKcwaxOmXMfSbn0bTD3Kn2Fg'}